# Document Question Answering System (RAG)

## Overview

This project builds a system that answers questions based on documents you provide. Instead of relying on what an AI model already knows, this system reads your document and finds relevant parts before generating an answer.

## Dataset

You can use any PDF or text file you already have:
- Class notes
- Your resume
- Research papers
- Books or articles

RAG works best with custom or private data that general AI models haven't seen.

## Objectives

By the end of this notebook, you'll have:
1. Built a RAG system that combines retrieval and generation
2. Created a system that answers questions about your own documents
3. Learned how modern AI question-answering works internally

## Key Concepts

RAG has three main parts:

**Retrieval** - Finds the relevant parts of your document. Uses embeddings (text turned into numbers) and similarity search.

**Augmentation** - Takes the retrieved text and attaches it to the question. This gives the AI actual evidence to work with.

**Generation** - The language model reads the question plus the retrieved context and writes an answer based on what it found.

## System Architecture

Here's what we're building:
1. Document Ingestion - Load PDF or text file
2. Text Chunking - Split text into smaller pieces
3. Embedding Creation - Turn chunks into vectors
4. Vector Database - Store embeddings for fast searching
5. Query Processing - Convert question to embedding
6. Context Retrieval - Find most similar chunks
7. Answer Generation - Generate answer using retrieved chunks

In [1]:
#  Install Required Libraries

!pip install -q pypdf sentence-transformers faiss-cpu transformers accelerate

In [2]:
import os
import re
import textwrap
import numpy as np
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

## Load Your Document

This function reads PDF or text files and returns their content as a string.

In [3]:
def load_document(file_path: str) -> str:
    """Read a PDF or text file and return its contents as a single string."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        reader = PdfReader(file_path)
        text = ""
        for page in reader.pages:
            page_text = page.extract_text() or ""
            text += page_text + "\n"
        return text

    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            return f.read()

    else:
        raise ValueError(f"Only PDF and TXT files are supported. Got '{ext}'")

##  Split Text into Chunks

Documents can be long. We break them into smaller overlapping chunks. The overlap ensures we don't cut important sentences in half.

In [4]:
def chunk_text(text: str, chunk_size: int = 180, chunk_overlap: int = 30):
    """Split text into overlapping chunks of approximately chunk_size words."""
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split(" ")

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - chunk_overlap

    return chunks

##  Create Embeddings

Convert each chunk into a vector of numbers that represents its meaning. Similar chunks will have similar numbers.

In [5]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(chunks: list) -> np.ndarray:
    """Convert a list of text chunks into embedding vectors."""
    embeddings = embedding_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
    return embeddings.astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

##  Build a Vector Database

Store embeddings in a searchable index using FAISS. This allows fast similarity search.

In [6]:
def build_vector_store(embeddings: np.ndarray):
    """Build a FAISS index for fast similarity search."""
    dim = embeddings.shape[1]
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)
    return index

##  Search for Relevant Chunks

Convert your question to an embedding and find the chunks with the most similar embeddings.

In [7]:
def retrieve(query: str, index, chunks: list, top_k: int = 3):
    """Find the top_k chunks most similar in meaning to the query."""
    query_embedding = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    scores, ids = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        results.append({
            "chunk": chunks[idx],
            "score": float(score),
            "chunk_id": int(idx)
        })
    return results

##  Generate the Answer

Combine the retrieved chunks with the question and ask a language model to generate an answer using only the provided context.

In [8]:
# ============================================
# ENHANCED FAST ANSWER GENERATION
# ============================================

def generate_answer(query: str, retrieved_chunks: list) -> str:
    """
    Extract answer directly from retrieved chunks with better quality.
    """
    if not retrieved_chunks:
        return "No relevant information found in the document."

    # Get the best matching chunk
    best_chunk = retrieved_chunks[0]["chunk"]

    # Also include second chunk if available for more context
    if len(retrieved_chunks) > 1:
        second_chunk = retrieved_chunks[1]["chunk"]
        full_text = best_chunk + ". " + second_chunk
    else:
        full_text = best_chunk

    # Clean up the text
    full_text = re.sub(r'\s+', ' ', full_text).strip()

    # Split into sentences
    sentences = full_text.split('. ')

    # Remove empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    # Return complete sentences
    if len(sentences) >= 4:
        return '. '.join(sentences[:4]) + '.'
    elif len(sentences) >= 3:
        return '. '.join(sentences[:3]) + '.'
    elif len(sentences) >= 2:
        return '. '.join(sentences[:2]) + '.'
    else:
        return full_text[:350] + "..."

## Handle Broad Questions

Questions like "What is this document about?" need a different approach. We sample chunks from across the entire document.

In [9]:
SUMMARY_KEYWORDS = [
    "main idea", "main point", "summary", "summarize", "summarise",
    "overview", "what is this document about", "what is the document about",
    "tl;dr", "gist", "what is this about", "in short",
]

def is_summary_query(question: str) -> bool:
    """Check if this is a broad, whole-document question."""
    q = question.lower()
    return any(keyword in q for keyword in SUMMARY_KEYWORDS)

def sample_chunks_across_document(chunks: list, max_chunks: int = 6) -> list:
    """Pick chunks spread evenly across the document."""
    if len(chunks) <= max_chunks:
        return chunks
    step = len(chunks) / max_chunks
    indices = sorted(set(int(i * step) for i in range(max_chunks)))
    return [chunks[i] for i in indices]

# ============================================
# FIXED: SUMMARIZE DOCUMENT (No get_generator)
# ============================================

def summarize_document(chunks: list) -> dict:
    """Summarize the document using chunks from across the whole text."""
    sample = sample_chunks_across_document(chunks, max_chunks=6)

    if chunks and chunks[0] not in sample:
        sample = [chunks[0]] + sample
    elif chunks and sample[0] != chunks[0]:
        sample = [chunks[0]] + [c for c in sample if c != chunks[0]]

    # Combine sampled chunks
    full_text = ". ".join(sample)

    # Clean up the text
    full_text = re.sub(r'\s+', ' ', full_text).strip()

    # Split into sentences
    sentences = full_text.split('. ')

    # Remove empty sentences
    sentences = [s.strip() for s in sentences if s.strip()]

    # Return first 5-6 sentences as summary
    if len(sentences) >= 6:
        summary = '. '.join(sentences[:6]) + '.'
    elif len(sentences) >= 4:
        summary = '. '.join(sentences[:4]) + '.'
    elif len(sentences) >= 2:
        summary = '. '.join(sentences[:2]) + '.'
    else:
        summary = full_text[:350] + "..."

    return {"answer": summary, "chunks_used": sample}


##  The Complete System

Combine everything into a simple class with two methods: `load()` and `ask()`.

In [10]:
class RAGSystem:
    def __init__(self, chunk_size: int = 180, chunk_overlap: int = 30, top_k: int = 3):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.top_k = top_k
        self.chunks = []
        self.index = None

    def load(self, file_path: str):
        """Load and process a document."""
        raw_text = load_document(file_path)
        self.chunks = chunk_text(raw_text, self.chunk_size, self.chunk_overlap)
        embeddings = embed_chunks(self.chunks)
        self.index = build_vector_store(embeddings)
        print(f"✅ Loaded '{file_path}' -> {len(self.chunks)} chunks ready.")

    def ask(self, question: str):
        """Answer a question about the loaded document."""
        if self.index is None:
            raise RuntimeError("Please call .load(file_path) first.")

        # Check if this is a broad question
        if is_summary_query(question):
            result = summarize_document(self.chunks)
            return {
                "question": question,
                "answer": self._clean_text(result["answer"]),
                "sources": [
                    {"chunk_id": None, "score": None,
                     "preview": textwrap.shorten(c, width=180)}
                    for c in result["chunks_used"]
                ],
            }

        # Regular question - retrieve relevant chunks
        retrieved = retrieve(question, self.index, self.chunks, self.top_k)

        # Generate clean answer - using simplified version
        answer = self._generate_clean_answer(question, retrieved)

        return {
            "question": question,
            "answer": answer,
            "sources": [
                {"chunk_id": r["chunk_id"], "score": round(r["score"], 3),
                 "preview": textwrap.shorten(r["chunk"], width=180)}
                for r in retrieved
            ],
        }

    def _generate_clean_answer(self, query: str, retrieved_chunks: list) -> str:
        """
        Generate a clean, natural language answer from retrieved chunks.
        Optimized for speed.
        """
        if not retrieved_chunks:
            return "I couldn't find information about that in the document."

        # Get best chunk only (skip combining to save time)
        best_chunk = retrieved_chunks[0]["chunk"]

        #  cleanup - it remove bullet points and extra spaces
        clean = best_chunk.replace('•', '').replace('', '').replace('●', '').replace('○', '')
        clean = re.sub(r'\s+', ' ', clean).strip()
        clean = re.sub(r'\d+\.\s*', '', clean)

        # Get first 3 sentences
        sentences = clean.split('. ')
        sentences = [s.strip() for s in sentences if s.strip() and len(s) > 10]

        # Return first 2-3 sentences
        if len(sentences) >= 3:
            answer = '. '.join(sentences[:3]) + '.'
        elif len(sentences) >= 2:
            answer = '. '.join(sentences[:2]) + '.'
        elif sentences:
            answer = sentences[0] + '.'
        else:
            answer = clean[:250] + "..."

        # Capitalize first letter
        if answer and answer[0].islower():
            answer = answer[0].upper() + answer[1:]

        return answer

    def _clean_text(self, text: str) -> str:
        """Quick text cleanup."""
        # Remove bullet points and extra spaces
        text = text.replace('•', '').replace('', '').replace('●', '').replace('○', '')
        text = re.sub(r'\s+', ' ', text).strip()
        text = re.sub(r'\d+\.\s*', '', text)
        return text

##  Upload Your Document

### Option A: Google Colab

In [11]:
try:
    from google.colab import files

    print("Please select a PDF or text file to upload:")
    uploaded = files.upload()
    uploaded_filename = next(iter(uploaded))

    rag = RAGSystem(chunk_size=180, chunk_overlap=30, top_k=3)
    rag.load(uploaded_filename)

except ModuleNotFoundError:
    print("Not in Google Colab. Please use the fallback option below.")

Please select a PDF or text file to upload:


Saving Artificial Intelligence in Robotics.pdf to Artificial Intelligence in Robotics (4).pdf


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Loaded 'Artificial Intelligence in Robotics (4).pdf' -> 7 chunks ready.


### Option B: Local Jupyter / Kaggle

In [12]:
# Uncomment and run this cell if you're not using Google Colab

# uploaded_filename = input("Enter the full path to your PDF or text file: ").strip()
# rag = RAGSystem(chunk_size=180, chunk_overlap=30, top_k=3)
# rag.load(uploaded_filename)

## Ask Questions

Ask questions about your document. Type 'exit' to quit.

In [12]:
print("Document loaded. Ask questions below. Type 'exit' to quit.\n")

while True:
    question = input("Your question: ").strip()

    if question.lower() in ("exit", "quit", ""):
        print("Session ended. Goodbye!")
        break

    result = rag.ask(question)

    print("\nAnswer:", result["answer"])
    print("\nSources used:")
    for s in result["sources"]:
        if s["chunk_id"] is not None:
            print(f"  [Chunk {s['chunk_id']}, Score: {s['score']}] {s['preview']}")
        else:
            print(f"  [Sampled from document] {s['preview']}")
    print("\n" + "-" * 60 + "\n")


Document loaded. Ask questions below. Type 'exit' to quit.

Your question: What is Artificial Intelligence in robotics

Answer: Artificial Intelligence in Robotics Artificial Intelligence (AI) in robotics represents one of the most transformative technological revolutions of the modern age. By combining the mechanical precision of robots with the cognitive power of AI, we are witnessing machines that can perceive, le arn, decide and act autonomously. Unlike traditional programmed robots that follow fixed instructions, AI-powered robots can adapt to new situations, analyze data in real -time and make intelligent decisions.

Sources used:
  [Chunk 0, Score: 0.648] Artificial Intelligence in Robotics Artificial Intelligence (AI) in robotics represents one of the most transformative technological revolutions of the modern age. By [...]
  [Chunk 5, Score: 0.646] robots to perform tasks independently, from navigation to problem-solving.  Perception: AI helps robots interpret sensory data, s

# Conclusion

## Project Summary

This project successfully implements a Retrieval-Augmented Generation (RAG) system that answers questions based on custom documents. The system was built from scratch using open-source tools and demonstrates how modern AI systems can combine retrieval and generation to provide accurate, grounded answers.

## What Was Achieved

The following objectives were accomplished:

### 1. Understanding RAG Concepts

- **Retrieval** - The system finds relevant document chunks using embeddings and vector similarity search
- **Augmentation** - Retrieved text is attached to questions as context
- **Generation** - Answers are generated based on actual document content, not from model memory

### 2. Building the Complete Pipeline

The system successfully implements all stages of a RAG pipeline:

| Stage | Implementation |
|-------|----------------|
| Document Ingestion | PDF and TXT file support using PyPDF |
| Text Chunking | Overlapping chunks with configurable size |
| Embedding Creation | Sentence Transformers (all-MiniLM-L6-v2) |
| Vector Database | FAISS for fast similarity search |
| Query Processing | Question embedding and retrieval |
| Answer Generation | Clean sentence extraction and formatting |

### 3. Key Features

The system provides:

- **Fast responses** - No language model loading, answers appear instantly
- **Clean answers** - Extracted sentences are cleaned and formatted
- **Source attribution** - Shows which chunks were used for each answer
- **Question type handling** - Both specific and broad questions are supported
- **Easy to use** - Simple interface: load document → ask questions

### 4. Testing and Validation

The system was tested with the document "Artificial Intelligence in Robotics" and successfully answered:

| Question Type | Example | Result |
|---------------|---------|--------|
| Definition | "What is AI in robotics?" | Complete, accurate answer |
| Technical | "How do neural networks work?" | Extracted from relevant section |
| Application | "How is AI used in healthcare?" | Retrieved healthcare information |
| Broad | "Summarize this document" | Combined multiple chunks |

## Technical Components Used

| Component | Purpose |
|-----------|---------|
| PyPDF | Reading PDF documents |
| Sentence Transformers | Creating text embeddings |
| FAISS | Vector similarity search |
| Regular Expressions | Text cleaning and processing |

## Key Learnings

Through this project, several important concepts were understood:

1. **Why retrieval matters** - Answer quality depends on finding the right chunks. If the system retrieves wrong information, the answer will be wrong regardless of how good the language model is.

2. **How embeddings work** - Text is converted to numbers that capture meaning. Similar chunks have similar numbers, enabling meaning-based search.

3. **Vector search** - FAISS enables fast similarity search over thousands of chunks, making the system scalable.

4. **Text processing** - Cleaning and extracting sentences from retrieved chunks is essential for readable answers.

5. **System design** - Building modular, reusable components makes the system easy to understand and extend.

## Strengths of the System

| Strength | Description |
|----------|-------------|
| No external API needed | Runs completely offline with free tools |
| Instant responses | No waiting for model downloads |
| Transparent answers | Sources are shown for verification |
| Works with any document | Supports PDF and TXT files |
| Beginner-friendly | Clean code with clear explanations |

## Limitations and Future Work

### Current Limitations

- Uses simple sentence extraction instead of language models
- Basic cleaning might not handle all formatting cases
- Limited to text-based answers

### Possible Improvements

- Add better sentence boundary detection
- Implement hybrid search (keyword + vector)
- Add metadata (page numbers, section headings)
- Try different embedding models for better accuracy
- Add GUI interface for easier use

